1) 컬렉션 목록 보기

In [8]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(allow_reset=True),
)

print(client.list_collections())

[Collection(name=rag_chunks)]


2) 컬렉션 일부 레코드 꺼내보기(documents/meta/ids)

In [9]:
col = client.get_collection("rag_chunks")  # app.py에서 쓰는 collection_name과 동일해야 함

# 앞에서부터 5개만
res = col.get(limit=5, include=["documents", "metadatas"])
print("ids:", res["ids"])
print("metas:", res["metadatas"])
print("docs:", [d[:120] for d in res["documents"]])  # 길면 앞부분만
print("total records:", col.count())  # 총 몇 개 들어있는지 확인

ids: ['rag_sample_document_ko.pdf|p1|c0', 'rag_sample_document_ko.pdf|p1|c1', 'rag_sample_document_ko.pdf|p2|c2']
metas: [{'chunk_id': 0, 'page': 1, 'source': 'rag_sample_document_ko.pdf'}, {'chunk_id': 1, 'page': 1, 'source': 'rag_sample_document_ko.pdf'}, {'chunk_id': 2, 'page': 2, 'source': 'rag_sample_document_ko.pdf'}]
docs: ['1\n RAG 실습용 샘플 문서 (한국어)\n이 문서는 Streamlit 기반 RAG(검색 증강 생성) 챗봇 실습에서 업로드/인덱싱/질의응답 흐름을\n테스트하기 위해 만든 예시 자료입니다.\n구성: (1) RAG 개념 요약', '스/인덱스입니다.\nTop-k 검색\n유사도가 높은 결과 k개를 가져오는 방식.\n재정렬(Re-ranking)\n1차 검색 결과를 더 정교한 모델(예: cross-encoder)로 다시 점수화하여 순서를 개선하는 단계.\n3', '2\nA. 한국어에 강한 임베딩 모델을 쓰고, 띄어쓰기/기호 정리를 최소한으로 하되 의미가 깨지지 않게\n전처리하세요.\n5. 테스트용 질의 예시\n1\n팬텀 촬영에서 SNR을 올리려면 어떤 파라미터를 조정해야 하나요?\n2\n']
total records: 3


3) 검색이 되는지 빠르게 확인(Query)

In [ ]:
import numpy as np
from openai import OpenAI

client_oai = OpenAI(api_key="...")  # 또는 환경변수 사용

def embed_one(text: str) -> list[float]:
    r = client_oai.embeddings.create(
        model="text-embedding-3-small",
        input=[text],
    )
    v = np.array(r.data[0].embedding, dtype=np.float32)
    v /= (np.linalg.norm(v) + 1e-12)
    return v.tolist()

qemb = embed_one("MMR이 뭐야?")

res = col.query(
    query_embeddings=[qemb],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(res["metadatas"][0])
print([d[:120] for d in res["documents"][0]])
print(res["distances"][0])

[{'source': 'rag_sample_document_ko.pdf', 'chunk_id': 2, 'page': 2}, {'source': 'rag_sample_document_ko.pdf', 'chunk_id': 0, 'page': 1}, {'chunk_id': 1, 'page': 1, 'source': 'rag_sample_document_ko.pdf'}]
['2\nA. 한국어에 강한 임베딩 모델을 쓰고, 띄어쓰기/기호 정리를 최소한으로 하되 의미가 깨지지 않게\n전처리하세요.\n5. 테스트용 질의 예시\n1\n팬텀 촬영에서 SNR을 올리려면 어떤 파라미터를 조정해야 하나요?\n2\n', '1\n RAG 실습용 샘플 문서 (한국어)\n이 문서는 Streamlit 기반 RAG(검색 증강 생성) 챗봇 실습에서 업로드/인덱싱/질의응답 흐름을\n테스트하기 위해 만든 예시 자료입니다.\n구성: (1) RAG 개념 요약', '스/인덱스입니다.\nTop-k 검색\n유사도가 높은 결과 k개를 가져오는 방식.\n재정렬(Re-ranking)\n1차 검색 결과를 더 정교한 모델(예: cross-encoder)로 다시 점수화하여 순서를 개선하는 단계.\n3']
[1.3716094493865967, 1.5550872087478638, 1.6070300340652466]
